# [저널 확장 · 2026-10-27] 공개 코드 재현 — FinRL EIIE의 평가 시작 상태 효과

**목적.** 다른 연구자의 공개 코드(FinRL `PortfolioOptimizationEnv` + `PolicyGradient` + `EIIE`, 커밋 `7e98df4`)를
**코드 수정 없이** 공식 데모 설정 그대로 돌리고, 평가 시작 보유만 바꿨을 때 결과가 얼마나 달라지는지 잰다.
바꾸는 것은 두 초기값뿐이다: 환경의 첫 보유 비중(`_actions_memory[0]`, `_final_weights[0]`)과 직전 행동 메모리(PVM) 초기값.
학습·평가 루프는 FinRL의 `train`/`test`를 그대로 호출한다.

**공식 데모 설정(그대로).** 브라질 주식 10종목(일봉, Yahoo), 종목별 MaxAbs 정규화, 학습 2011-01-01~2019-12-30,
테스트 2020·2021·2022 각 1년, 수수료 0.25%(TRF), time_window 50, 특징 close/high/low, lr 0.01, 40 에피소드,
테스트 중 온라인 학습(10스텝마다). 데모에는 시드가 없으므로 시드 {0, 1, 2}로 3번 학습한다.

**사전 등록(실행 전 커밋).**

| 항목 | 값 |
|---|---|
| 시작 보유 | 현금 100%(FinRL 기본) / 주식 등가중 100% / 워밍업(측정 시작 60거래일 전부터 현금으로 실행) |
| 온라인 학습 | 켜짐(데모 원래 방식, 10스텝마다) / 꺼짐(시작 상태 효과만 분리) |
| 관성 추정 | 학습 구간 20개 시점에서 현금·등가중 시작으로 60거래일 실행(온라인 꺼짐) → 노출(1 − 현금 비중) 차이의 평균을 k = 0~59에서 log 선형 회귀 → α_eff = 1 − exp(기울기). x̄ = 같은 시점 워밍업 실행의 평균 노출 |
| 예측 | 테스트 연도별(온라인 꺼짐) 예측 = (x₀ − x̄) · Σ (1 − α_eff)^k · log(1 + b_k), b = 10종목 등가중 일간 수익, x₀ = 0(현금)·1(등가중). 테스트 실행 전에 확정 |
| H6 학습된 관성 | 시드 3개 중 2개 이상에서 α_eff < 0.5 |
| H7 공식 | H6 충족 시, α_eff < 0.5인 시드의 (연도 × 시작) 쌍을 합쳐 관측(시작 − 워밍업 log 성장) vs 예측 상관 ≥ 0.8, 기울기 ∈ [0.7, 1.3] |
| H8 실무 영향 | 데모 원래 방식(온라인 켜짐)에서 Δ = log 성장(등가중 시작) − log 성장(현금 시작), 9개 셀(시드 × 연도)의 \|Δ\| 중앙값 ≥ 1.0 로그%p 이면 '보고 수익에 실질적 영향'. 데모 벤치마크(매 스텝 등가중) 대비 초과수익의 부호가 바뀌는 셀 수도 보고 |

- 데이터는 실행 시점에 Yahoo에서 받으므로 데모 작성 시점과 미세하게 다를 수 있다. 받은 원자료를 Drive에 저장한다.
- **`런타임 > 모두 실행`.** CPU로 약 1~2시간. 끊기면 다시 `모두 실행` — 학습된 시드·끝난 평가는 건너뛴다.
- 결과: Drive `data/finrl_replication/` (`raw_prices.csv`, `alpha.csv`, `windows.csv`, `protocol.csv`)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gym==0.26.2 torch_geometric quantstats yfinance stable-baselines3

In [ ]:
# FinRL 원본 파일(커밋 7e98df4)을 수정 없이 받아 패키지로 쓴다
import os, sys, urllib.request, pathlib
COMMIT = '7e98df461cbc7772f16a545a2f13b521e5d0b8f4'
RAW = f'https://raw.githubusercontent.com/AI4Finance-Foundation/FinRL/{COMMIT}/finrl/'
PKG = pathlib.Path('/content/finrl_po'); PKG.mkdir(exist_ok=True); (PKG / '__init__.py').write_text('')
for f in ['agents/portfolio_optimization/algorithms.py', 'agents/portfolio_optimization/architectures.py',
          'agents/portfolio_optimization/utils.py', 'agents/portfolio_optimization/models.py',
          'meta/env_portfolio_optimization/env_portfolio_optimization.py', 'meta/preprocessor/yahoodownloader.py']:
    urllib.request.urlretrieve(RAW + f, PKG / pathlib.Path(f).name)
sys.path.insert(0, '/content')
import warnings, logging; warnings.filterwarnings('ignore'); logging.getLogger('matplotlib.font_manager').disabled = True
import random, numpy as np, pandas as pd, torch
from sklearn.preprocessing import MaxAbsScaler
from finrl_po.yahoodownloader import YahooDownloader
from finrl_po.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl_po.models import DRLAgent
from finrl_po.architectures import EIIE
from finrl_po import utils as fu
OUT = pathlib.Path('/content/drive/MyDrive/졸업프로젝트/data/finrl_replication'); OUT.mkdir(parents=True, exist_ok=True)
MDIR = pathlib.Path('/content/drive/MyDrive/졸업프로젝트/models/finrl_replication'); MDIR.mkdir(parents=True, exist_ok=True)
device = 'cpu'
print('FinRL', COMMIT[:7], '로드 완료')

In [ ]:
# 데이터 — 공식 데모와 같은 종목·기간. GroupByScaler(by='tic', MaxAbsScaler)와 같은 처리(종목별 최대 절댓값으로 나눔)
TOP_BRL = ["VALE3.SA", "PETR4.SA", "ITUB4.SA", "BBDC4.SA", "BBAS3.SA", "RENT3.SA", "LREN3.SA", "PRIO3.SA", "WEGE3.SA", "ABEV3.SA"]
raw_path = OUT / 'raw_prices.csv'
if raw_path.exists():
    raw = pd.read_csv(raw_path)
else:
    raw = YahooDownloader(start_date='2011-01-01', end_date='2022-12-31', ticker_list=TOP_BRL).fetch_data()
    raw.to_csv(raw_path, index=False)
num = [c for c in raw.columns if c not in ('date', 'tic', 'day') and pd.api.types.is_numeric_dtype(raw[c])]
norm = raw.copy()
for t, g in raw.groupby('tic'):
    norm.loc[g.index, num] = MaxAbsScaler().fit_transform(g[num])
df = norm[['date', 'tic', 'close', 'high', 'low']].copy(); df['date'] = df['date'].astype(str)
dates = sorted(df['date'].unique())
def sl(d0, d1): return df[(df['date'] >= d0) & (df['date'] < d1)]
df_train = sl('2011-01-01', '2019-12-31')
YEARS = {2020: sl('2020-01-01', '2020-12-31'), 2021: sl('2021-01-01', '2021-12-31'), 2022: sl('2022-01-01', '2022-12-31')}
N, TW, FEE, W, H = len(TOP_BRL), 50, 0.0025, 60, 60
print('종목', df.tic.nunique(), '| 학습', df_train.date.nunique(), '일 | 연도별', {y: d.date.nunique() for y, d in YEARS.items()})

In [ ]:
# 시작 보유 제어 — FinRL 코드는 그대로 두고 초기값 두 곳만 바꾼다
W0 = {'cash': np.array([1] + [0] * N, dtype=np.float32), 'ew': np.array([0] + [1 / N] * N, dtype=np.float32)}
_START = ['cash']
_orig_pvm_reset = fu.PVM.reset
def _pvm_reset(self):
    _orig_pvm_reset(self)
    self.memory = [W0[_START[0]].copy()] * (self.capacity + 1)
fu.PVM.reset = _pvm_reset

class StartEnv(PortfolioOptimizationEnv):
    def _reset_memory(self):
        super()._reset_memory()
        self._actions_memory[0] = W0[_START[0]].copy(); self._final_weights[0] = W0[_START[0]].copy()

def make_env(d):
    return StartEnv(d, initial_amount=100000, comission_fee_pct=FEE, time_window=TW, features=['close', 'high', 'low'], normalize_df=None)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def run(model, policy, d, start, online, seed=0):
    # FinRL model.test 그대로 실행. 반환: (스텝별 가치, 스텝별 노출 = 1 − 현금 비중)
    _START[0] = start; seed_all(seed)
    env = make_env(d)
    DRLAgent.DRL_validation(model, env, policy=policy, online_training_period=10 if online else 10**9)
    v = np.asarray(env._asset_memory['final'], dtype=np.float64)
    x = np.asarray([1 - float(a[0]) for a in env._actions_memory[1:]])
    _START[0] = 'cash'
    return v, x

def window(d_all_dates, first_decision, extra_before):
    # first_decision 날짜에 첫 결정을 내리도록, 그 앞 TW−1 + extra_before 거래일을 포함한 구간
    p = d_all_dates.index(first_decision)
    return df[(df['date'] >= d_all_dates[p - (TW - 1) - extra_before])]
print('준비 완료')

In [ ]:
%%time
# 학습 — 공식 데모 설정, 시드 3개. 있으면 건너뛴다
SEEDS = [0, 1, 2]
for s in SEEDS:
    p = MDIR / f'policy_EIIE_s{s}.pt'
    if p.exists():
        print('있음:', p.name); continue
    seed_all(s)
    model = DRLAgent(make_env(df_train)).get_model('pg', device, {'lr': 0.01, 'policy': EIIE}, {'k_size': 3, 'time_window': TW})
    DRLAgent.train_model(model, episodes=40)
    torch.save(model.train_policy.state_dict(), p); print('저장:', p.name, flush=True)

In [ ]:
%%time
# 관성 추정(학습 구간) → 테스트 예측 확정 → 관측(온라인 꺼짐) + 데모 방식(온라인 켜짐)
AP, WP, PP = OUT / 'alpha.csv', OUT / 'windows.csv', OUT / 'protocol.csv'
A = pd.read_csv(AP).to_dict('records') if AP.exists() else []
Wr = pd.read_csv(WP).to_dict('records') if WP.exists() else []
Pr = pd.read_csv(PP).to_dict('records') if PP.exists() else []
DONE = {int(r['seed']) for r in A}
Wr = [r for r in Wr if int(r['seed']) in DONE]; Pr = [r for r in Pr if int(r['seed']) in DONE]
close_wide = raw.pivot(index='date', columns='tic', values='close'); close_wide.index = close_wide.index.astype(str)

def basket_log(first_decision, n_steps):
    # 결정 시점 t의 보유가 받는 수익 = close[t+1]/close[t] − 1 (FinRL price_variation과 같은 정렬)
    i = dates.index(first_decision)
    c = close_wide.loc[dates[i:i + n_steps + 1]]
    return np.log1p((c.iloc[1:].to_numpy() / c.iloc[:-1].to_numpy() - 1).mean(axis=1))

tr_dates = sorted(df_train['date'].unique())
for s in SEEDS:
    if s in DONE:
        print('건너뜀(완료): seed', s); continue
    policy = EIIE(time_window=TW, device=device); policy.load_state_dict(torch.load(MDIR / f'policy_EIIE_s{s}.pt'))
    model = DRLAgent(make_env(df_train)).get_model('pg', device, {'lr': 0.01, 'policy': EIIE}, {'k_size': 3, 'time_window': TW})
    gaps, xs = [], []
    for p in np.linspace(TW - 1 + W, len(tr_dates) - H - 2, 20).astype(int):
        d_on = df[(df['date'] >= tr_dates[p - (TW - 1)]) & (df['date'] <= tr_dates[p + H])]
        _, x_ew = run(model, policy, d_on, 'ew', False); _, x_c = run(model, policy, d_on, 'cash', False)
        d_w = df[(df['date'] >= tr_dates[p - (TW - 1) - W]) & (df['date'] <= tr_dates[p + H])]
        _, x_w = run(model, policy, d_w, 'cash', False)
        m = min(len(x_ew), len(x_c), H); gaps.append(np.abs(x_ew[:m] - x_c[:m])); xs.append(x_w[W:].mean())
    g = np.mean(np.stack([q[:min(map(len, gaps))] for q in gaps]), axis=0); k = np.arange(len(g)); ok = g > 1e-4
    slope = np.polyfit(k[ok], np.log(g[ok]), 1)[0] if ok.sum() > 5 else 0.0
    a_eff, xbar = float(np.clip(1 - np.exp(slope), 1e-4, 1.0)), float(np.mean(xs))
    print(f'seed {s}: α_eff={a_eff:.4f}, x̄={xbar:.3f}, 초기 노출 차이 {g[0]:.3f}', flush=True)
    rows_w, rows_p = [], []
    yd = {y: sorted(d['date'].unique()) for y, d in YEARS.items()}
    preds = {}
    for y, dl in yd.items():   # 예측을 먼저 모두 확정
        n = len(dl) - TW
        bl = basket_log(dl[TW - 1], n)
        for st, x0 in (('cash', 0.0), ('ew', 1.0)):
            preds[(y, st)] = float((x0 - xbar) * ((1 - a_eff) ** np.arange(len(bl)) * bl).sum())
    for y, dl in yd.items():
        d_y = YEARS[y]; first = dl[TW - 1]
        v_w, _ = run(model, policy, window(dates, first, W).query('date <= @dl[-1]'), 'cash', False)
        g_w = np.log(v_w[-1] / v_w[W])
        for st in ('cash', 'ew'):
            v, _ = run(model, policy, d_y, st, False)
            rows_w.append({'seed': s, '연도': y, '시작': st, '예측': preds[(y, st)], '관측': float(np.log(v[-1] / v[0]) - g_w)})
            v_on, _ = run(model, policy, d_y, st, True, seed=s)
            rows_p.append({'seed': s, '연도': y, '시작': st, '온라인 log 성장': float(np.log(v_on[-1] / v_on[0]))})
        # 데모 벤치마크(매 스텝 등가중, 현금 시작) — 데모 코드와 같은 방식
        _START[0] = 'cash'; env = make_env(d_y); env.reset(); term = False
        while not term:
            _, _, term, _ = env.step([0] + [1 / N] * N)
        rows_p.append({'seed': s, '연도': y, '시작': '벤치마크(등가중)', '온라인 log 성장': float(np.log(env._asset_memory['final'][-1] / env._asset_memory['final'][0]))})
    Wr += rows_w; Pr += rows_p; A.append({'seed': s, 'α_eff': a_eff, 'x̄': xbar, '초기 노출 차이': float(g[0])})
    pd.DataFrame(Wr).to_csv(WP, index=False); pd.DataFrame(Pr).to_csv(PP, index=False); pd.DataFrame(A).to_csv(AP, index=False)
print('저장 완료')

In [ ]:
# 사전 등록 판정
A_ = pd.DataFrame(A); W_ = pd.DataFrame(Wr); P_ = pd.DataFrame(Pr)
print(A_.round(4).to_string(index=False))
inert = A_[A_['α_eff'] < 0.5]['seed'].tolist()
h6 = len(inert) >= 2
print(f"\n[H6] α_eff < 0.5 시드 {len(inert)}/3 → {'지지' if h6 else '지지 안 됨'}")
if h6:
    Wi = W_[W_['seed'].isin(inert)]
    r = np.corrcoef(Wi['예측'], Wi['관측'])[0, 1]; sl_ = np.polyfit(Wi['예측'], Wi['관측'], 1)[0]
    print(f"[H7] 쌍 {len(Wi)}개: 상관 {r:.3f}, 기울기 {sl_:.2f} → {'지지' if r >= 0.8 and 0.7 <= sl_ <= 1.3 else '지지 안 됨'}")
else:
    print('[H7] 판정 안 함(H6 미충족)')
print((W_.assign(예측=W_['예측'] * 100, 관측=W_['관측'] * 100)).round(2).to_string(index=False))
pv = P_.pivot_table(index=['seed', '연도'], columns='시작', values='온라인 log 성장')
pv['Δ(등가중−현금) 로그%p'] = (pv['ew'] - pv['cash']) * 100
pv['초과(현금 시작)'] = (pv['cash'] - pv['벤치마크(등가중)']) * 100; pv['초과(등가중 시작)'] = (pv['ew'] - pv['벤치마크(등가중)']) * 100
med = pv['Δ(등가중−현금) 로그%p'].abs().median(); flips = int((np.sign(pv['초과(현금 시작)']) != np.sign(pv['초과(등가중 시작)'])).sum())
print(pv.round(2).to_string())
print(f"\n[H8] |Δ| 중앙값 {med:.2f} 로그%p → {'실질적 영향' if med >= 1.0 else '실질적 영향 아님'} | 초과수익 부호가 바뀐 셀 {flips}/9")